## Enriching stock market data using Open AI API 

The Nasdaq-100 is a stock market index made up of 101 equity securities issued by 100 of the largest non-financial companies listed on the Nasdaq stock exchange. It helps investors compare stock prices with previous prices to determine market performance.

In this project you are provided with two CSV files containing Nasdaq-100 stock information:
- _**nasdaq100_CA.csv**_: contains information about companies in the index such as symbol, name, etc. For this analysis, only companies headquartered in California have been selected.
- _**nasdaq100_price_change.csv**_: contains price changes per stock across periods including (but not limited to) one day, five days, one month, six months, one year, etc.

As an AI developer, you will leverage the OpenAI API to classify companies into sectors and produce a summary of sector and company performance for this year, for the companies in the index that are headquartered in California.

# CSV with Nasdaq-100 stock data

In this project, you have available two CSV files `nasdaq100_CA.csv` and `nasdaq100_price_change.csv`.

## nasdaq100_CA.csv

```py
symbol,name,headQuarter,dateFirstAdded,cik,founded
AAPL,Apple Inc.,"Cupertino, CA",,0000320193,1976-04-01
ABNB,Airbnb,"San Francisco, CA",,0001559720,2008-08-01
ADBE,Adobe Inc.,"San Jose, CA",,0000796343,1982-12-01
...
```

## nasdaq100_price_change.csv

```py
symbol,1D,5D,1M,3M,6M,ytd,1Y,3Y,5Y,10Y,max
AAPL,-1.7254,-8.30086,-6.20411,3.042,15.64824,42.99992,8.47941,60.96299,245.42031,976.99441,139245.53954
ABNB,2.1617,-2.21919,9.88336,19.43286,19.64241,68.66902,23.64013,-1.04347,-1.04347,-1.04347,-1.04347
ADBE,0.5409,-1.77817,9.16191,52.0465,38.01522,57.22723,21.96206,17.83037,109.05718,1024.69214,251030.66399
ADI,0.9291,-4.03352,2.58486,3.65887,5.01602,17.02062,8.09735,63.42847,92.81874,286.77518,26012.63736
...
```

In [2]:
# Start your code here!
import os
import pandas as pd
from openai import OpenAI

# API instance of OpenAI initiated and CSV files read
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
nasdaq100_ca = pd.read_csv("nasdaq100_CA.csv")
pchange = pd.read_csv("nasdaq100_price_change.csv")

#symbol added into nasdaq100 dataframe from pchange.
nasdaq100_ca = nasdaq100_ca.merge(pchange[["symbol", "ytd"]], on="symbol", how="inner")

# This for loop loops around companies, asking the AI the prompt below and then storing it in AInswer.
for company in nasdaq100_ca["symbol"]:
    prompt = f'''I need you to classify {company} into one the next sectors. Do NOT answer anything besides the sector name: Technology, Consumer Cyclical, Industrials, Utilities, Healthcare, Communication, Energy, Consumer Defensive, Real Estate, Financial.
'''
    AInswer = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{ "role": "user", "content": prompt}],
        temperature=0.0,
    )
    sector = AInswer.choices[0].message.content
    nasdaq100_ca.loc[nasdaq100_ca["symbol"] == company, "Sector"] = sector #sector column stored for such company.

nasdaq100_ca["Sector"].value_counts() #number of sectors counted

# The propmt below is asked and its answer is stored in variable AInswer
prompt = f'''Give me a summary of information about Nasdaq-100's YTD stock performance from companies with their headquarters located in CA, recommending top two sectors and top three companies per sector. Use this data: {nasdaq100_ca} 
'''

AInswer = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{ "role": "user", "content": prompt}],
        temperature=0.0,
    )

# Store the output as a variable and print the recommendations
stock_recommendations = AInswer.choices[0].message.content
print(stock_recommendations)

The top two sectors with the best YTD stock performance from companies headquartered in California are Technology and Real Estate.

Top three companies in the Technology sector:
1. Nvidia (NVDA) - YTD performance: 217.27%
2. Meta Platforms (META) - YTD performance: 153.78%
3. Lam Research (LRCX) - YTD performance: 70.25%

Top three companies in the Real Estate sector:
1. Airbnb (ABNB) - YTD performance: 68.67%
2. Lucid Motors (LCID) - YTD performance: 3.89%

Overall, the Technology sector has shown the best YTD stock performance among companies with headquarters in California.
